# Anime Stitch Pipeline — Benchmark Analysis

This notebook walks through the ASP benchmark corpus (97 tests in `backend/benchmark/`),
visualises per-test quality metrics, and identifies failure clusters.

**Prerequisites**:
```bash
source .venv/bin/activate
pip install matplotlib pandas seaborn
```

**Run benchmarks first** (if no cached results exist):
```bash
pytest backend/test/anim/ --skip-gpu -q
python backend/benchmark/run_all.py
```

In [ ]:
import json
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid', palette='muted')

RESULTS_DIR = pathlib.Path('backend/benchmark/results')

# Load all JSON result files into a single DataFrame
records = []
for p in sorted(RESULTS_DIR.glob('*.json')):
    try:
        data = json.loads(p.read_text())
        if isinstance(data, list):
            records.extend(data)
        else:
            records.append(data)
    except (json.JSONDecodeError, OSError):
        pass

df = pd.DataFrame(records)
print(f'Loaded {len(df)} benchmark records from {RESULTS_DIR}')
df.head()

## 1. Metric Overview

The ASP pipeline emits these quality metrics per test case:

| Metric | Range | Lower is better? | Description |
|---|---|---|---|
| `ssim` | [0, 1] | No (higher = better) | Structural similarity vs ground truth |
| `aligned_ssim` | [0, 1] | No | SSIM after ECC alignment (S25 fix) |
| `ghosting_score` | [0, 100] | Yes | Original ghost detector (used by GhostGate) |
| `ghosting_siqe` | [0, 100] | Yes | FFT autocorrelation ghost detector (S35) |
| `seam_visibility` | ≥ 0 | Yes | Adjacent-row luminance jump at seam |
| `rlhf_score` | [0, 1] | Yes | RLHF reward model score (S29, random weights until feedback collected) |
| `rlhf_flagged` | bool | — | True when rlhf_score > 0.6 |
| `method` | str | — | `'asp'` or `'scans'` (fallback) |

In [ ]:
numeric_cols = ['ssim', 'aligned_ssim', 'ghosting_score', 'ghosting_siqe',
                'seam_visibility', 'rlhf_score']
available = [c for c in numeric_cols if c in df.columns]

desc = df[available].describe().T
desc['pass_rate'] = None  # filled below for binary-threshold metrics
display(desc)

## 2. SSIM Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['ssim', 'aligned_ssim']):
    if col not in df.columns:
        ax.set_visible(False)
        continue
    sns.histplot(df[col].dropna(), bins=20, ax=ax, kde=True)
    ax.axvline(df[col].median(), color='red', linestyle='--', label=f'median={df[col].median():.3f}')
    ax.set_title(col)
    ax.legend()

fig.suptitle('Structural Similarity (higher = better)')
plt.tight_layout()
plt.show()

## 3. Ghosting Score Distribution

The **GhostGate** in the pipeline uses `ghosting_score` (original) as its live gate.
`ghosting_siqe` (S35, FFT autocorrelation) is a more sensitive detector but has not yet
replaced the gate — calibration is in progress.

In [ ]:
ghost_cols = [c for c in ['ghosting_score', 'ghosting_siqe'] if c in df.columns]
if ghost_cols:
    fig, ax = plt.subplots(figsize=(9, 4))
    for col in ghost_cols:
        sns.kdeplot(df[col].dropna(), ax=ax, label=col, fill=True, alpha=0.3)
    ax.axvline(30, color='orange', linestyle=':', label='ghost_siqe flag threshold (30)')
    ax.set_xlabel('Ghost score')
    ax.set_title('Ghosting score distributions (lower = less ghosting)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No ghosting columns found in results.')

## 4. ASP vs SCANS Fallback Rate

Tests where `method == 'scans'` used the lightweight fallback instead of the full 13-stage pipeline.
The target fallback rate is ≤ 5% (currently 4/96 genuine fallbacks after S11 fixes).

In [ ]:
if 'method' in df.columns:
    counts = df['method'].value_counts()
    fig, ax = plt.subplots(figsize=(5, 4))
    counts.plot.bar(ax=ax, color=['steelblue', 'tomato'])
    ax.set_title('Pipeline used per test')
    ax.set_ylabel('Test count')
    for i, v in enumerate(counts):
        ax.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
    pct_scans = counts.get('scans', 0) / counts.sum() * 100
    ax.set_xlabel(f'SCANS fallback rate: {pct_scans:.1f}%')
    plt.tight_layout()
    plt.show()
else:
    print('No "method" column found — skipping fallback rate chart.')

## 5. Failure Taxonomy

Failures are classified into four clusters based on the benchmark report:

| Cluster | Root cause | Metric signature |
|---|---|---|
| **Ghost / double-edge** | Pose gap at seam + wide feather | `ghosting_siqe > 30` |
| **Hard seam step** | Colour discontinuity at boundary | `seam_visibility > 25` |
| **Low structural similarity** | Misaligned panels | `ssim < 0.70` |
| **SCANS fallback** | Pipeline validation failure | `method == 'scans'` |

In [ ]:
fail_flags = pd.DataFrame(index=df.index)

if 'ghosting_siqe' in df.columns:
    fail_flags['ghost'] = df['ghosting_siqe'] > 30
if 'seam_visibility' in df.columns:
    fail_flags['hard_seam'] = df['seam_visibility'] > 25
if 'ssim' in df.columns:
    fail_flags['low_ssim'] = df['ssim'] < 0.70
if 'method' in df.columns:
    fail_flags['scans_fallback'] = df['method'] == 'scans'

if not fail_flags.empty:
    summary = fail_flags.sum().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(7, 4))
    summary.plot.barh(ax=ax, color='tomato')
    ax.set_xlabel('Number of tests')
    ax.set_title('Failure cluster counts')
    plt.tight_layout()
    plt.show()
    print('\nTests failing in multiple clusters:')
    display(fail_flags[fail_flags.sum(axis=1) > 1])
else:
    print('No failure flag columns available.')

## 6. Metric Correlation Heatmap

Understanding which metrics co-vary helps prioritise which gates to tighten.

In [ ]:
corr_cols = [c for c in ['ssim', 'aligned_ssim', 'ghosting_score', 'ghosting_siqe',
                          'seam_visibility', 'rlhf_score'] if c in df.columns]
if len(corr_cols) >= 2:
    corr = df[corr_cols].corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Metric correlation matrix')
    plt.tight_layout()
    plt.show()
else:
    print('Need at least 2 numeric metric columns for a correlation heatmap.')

## 7. Per-Test Summary Table

Full table of all test results, sortable in Jupyter.

In [ ]:
display_cols = ['test_id', 'method'] + [c for c in numeric_cols if c in df.columns] + ['rlhf_flagged']
display_cols = [c for c in display_cols if c in df.columns]

styled = (
    df[display_cols]
    .sort_values('test_id' if 'test_id' in df.columns else display_cols[0])
    .style
    .background_gradient(subset=[c for c in ['ssim', 'aligned_ssim'] if c in display_cols],
                         cmap='RdYlGn')
    .background_gradient(subset=[c for c in ['ghosting_score', 'ghosting_siqe', 'seam_visibility']
                                  if c in display_cols], cmap='RdYlGn_r')
    .format({c: '{:.3f}' for c in numeric_cols if c in display_cols})
)
display(styled)